In [6]:
import pandas as pd
# pd.set_option('display.max_colwidth', None) # if you want to view the full json blob in the printed dataframe, use this

#These file locations are local, so they would need to be updated if running from a different machine. To update the Power BI report, upload the file output from this script here and refresh the semantic model: https://usdagcc.sharepoint.com/:f:/r/sites/fs-cio-aicouncil/Shared%20Documents/General/Use%20Cases/EMC/NLP_Public_Comments?csf=1&web=1&e=rmcD2a 
file_path = r'C:\Users\ChristinePayton\OneDrive - USDA\Documents\Data\NLP_Public_Comments\letters_2016_2017.csv'
supplemental_file = r'C:\Users\ChristinePayton\OneDrive - USDA\Documents\Data\NLP_Public_Comments\additional_letter_info.csv'
output_path = r'C:\Users\ChristinePayton\OneDrive - USDA\Documents\Data\NLP_Public_Comments\letters_ai_extraction_full_2016_2017.csv'

# get csv
#the different CSV years are encoded differently so you need to pass a diff parameter depending on which year file you're reading
#2019-2022 = cp1252
#2016-2018 = ANSI
#you can check encoding on CSV files by opening it in notepad, it'll say what it is in the bottom right corner
df = pd.read_csv(file_path, encoding="ANSI") #these files are encoded strangely, have to pass encoding value
supplemental = pd.read_csv(supplemental_file, encoding="cp1252") #this file contains the 'unique' flag we're using to filter out the noise

# left join letters with supplemental file on the 'Letter Id' field
df = pd.merge(df, supplemental, on='Letter Id', how='left')

letter_types = ['Unique'] #the letter types we want from the supplemental data file, can comma separate multiple

#Filter for only 'unique' values (this dataset is 90% generated form duplicate data, don't want to use tokens on that) and letters that are not essentially null (these cause errors)
df = df[(df['Letter Type'].isin(letter_types)) & (df['Letter Text'].str.len() > 10)]

df = df.drop_duplicates(subset=['Author Name', 'Letter Text']) #there are some letters that are empty or things like "see attachments" from the same person
#df = df.sample(n=10) #get a random small number of rows for testing, remove this line later to run on full csv

df


,ProjectId,Project Name,Comment Period Id,Comment Period Type,Comment Period Name,Comment Period Start Year,Author Name,Author ZipCode,Letter Id,Letter Sequence Number,Letter Text,Letter Submission Date,Form Set,Letter Type,Delivery Type
3,1214,2016 Renewal of Existing Permits for Outfitter...,1476,Scoping,NaN,2016,AJ Myers,97754,1151680,1,Length of permit: I am in favor of permits be...,2016-02-25 22:43:32.0000000,NaN,Unique,CARA Web-portal
4,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,mark perkins,85048,1253492,1,"Hello, great to read about the 4 Forest Restor...",2016-07-23 00:11:36.0000000,NaN,Unique,CARA Web-portal
5,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,Stephen Clark,85381,1289130,2,"August 8, 2016\n\nAnnette Fredette, 4FRI Plann...",2016-08-08 07:00:00.0000000,NaN,Unique,CARA Web-portal
6,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,Fred Gaudet,85067,1295026,3,Also see attachment for pdf of letter\n\nAugus...,2016-08-09 19:59:00.0000000,NaN,Unique,CARA Web-portal
7,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,Nate Reisner,86001,1313126,4,"4 FRI Project Team: Annette Fredette, 4 FRI Pl...",2016-10-06 00:00:00.0000000,NaN,Unique,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366342,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,LeeAnn Bennett,66049-2029,1373906,2,"Forest Service,\n\nI have no objection to this...",2017-04-29 02:36:43.0000000,NaN,Unique,CARA Web-portal
366343,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,Scott Amos,NaN,1375429,3,It's not very often that I have something posi...,2017-04-28 11:00:00.0000000,NaN,Unique,Email
366345,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,Joel Fields,NaN,1375431,5,We have a summer cabin in Yellow Pine and have...,2017-05-01 11:00:00.0000000,NaN,Unique,Email
366346,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,Johnna Sandow,NaN,1375432,6,I saw this has a June 2017 implementation date...,2017-04-28 11:00:00.0000000,NaN,Unique,Email


In [7]:
#GPT 3.5 ITERATE OVER DATAFRAME - I swear this used to require VPN but doesn't seem to anymore - so if you get a 400 error, try VPN
import os
from openai import AzureOpenAI
import json

#Azure OpenAI connection info
client = AzureOpenAI(
    azure_endpoint = "https://oai-nonprd-openai-poc-01.openai.azure.com", 
    api_key = os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-02-15-preview"
)

#This is what gets sent to Azure OpenAI - system message and user message
def completion_iteration(value):
    try:
        # This is the structure for GPT 3.5, other models may have different syntax
        dynamic_message_text = [
            {"role": "system", "content": "You are a feedback analyst. You parse and extract from submitted letters to help organize and structure the data."},
            {"role": "user", "content": user_prompt + json.dumps(example_json) + "Here is the letter text to analyze: " + value}
        ]
        
        completion = client.chat.completions.create(
            model="BASE-gpt-35-turbo",
            messages=dynamic_message_text,  # Use the dynamically constructed message
            temperature=0.6,
            max_tokens=800,
            top_p=0.95,
            frequency_penalty=0,
            presence_penalty=0,
            stop=None
        )
        return completion.choices[0].message.content #this returns just the response text
    except Exception as e:
        #commenting this out for now as it's super verbose and massively scrolls the thing down
        #print(f"Error processing input '{value}': {str(e)}")
        return None
#These were the suggested categories for response triage
#categories as text, not array, because we are passing in message string. You can replace these values with others to tailor the categorization
response_categories = "Air Quality, Botany, Climate Change, Cultural/Heritage, Facilities, FireFuels, Fisheries, Hydrology, Lands/Special Uses, Minerals/Geology, NEPA/Proj Development, Other/Misc, Public Engagement, Range/Weeds, Recreation, Roadless, Silviculture/Veg, SocioEconomic, Soils, Transportation, Visuals, Wilderness, Wildlife"

#this is the actual prompt, we ask for it to respond with JSON so that it is semi-structured and can be put into a dataframe easily
user_prompt = "Extract the following information: Sentiment (required, -1 to 1, where -1 is extremely negative and 1 is extremely positive), SentimentConfidence (decimal, 0-1, represents the confidence level of your sentiment number), Category (required, single value, choices are: " + response_categories + ". CitingLaw (boolean, return True if the text implies that a law is being broken or is referencing a law), BriefSummary (required, 1-2 sentence summary of the letter), SpecificFeedback (boolean, required, true if a specific resolution to their issue is identified), ProposedResolution (summarization of what the submitter thinks will resolve their issue). Your response should be ONLY the JSON analysis, no other text. Here is an example response: "

#Giving the AI an example is required here to get a consistent output
example_json = {
  "Sentiment": -0.5,
  "SentimentConfidence": .8,
  "Category": "Hydrology",
  "CitingLaw": True,
  #"Tags": ["water quality", "pollution", "regulation"], #not doing tags anymore, replaced with citing law at request
  "BriefSummary": "The letter expresses concerns over river pollution impacting community health and calls for stricter regulatory oversight and sustainable practices.",
  "SpecificFeedback": True,
  "ProposedResolution":"Stricter regulation on industry waste dumping."
  #you could also have it propose a response that people could review/use as a starting point here e.g.:
  #"SuggestedResponse":"Thank you for your thoughtful feedback regarding the dumping regulations. We share your concern about the need for robust measures to protect our environment and public health. Please know that your comments are invaluable to us and contribute significantly to our ongoing review process aimed at strengthening our regulations. We are actively working with various stakeholders, including environmental experts and community leaders, to ensure our policies effectively address these concerns while promoting sustainable practices. To stay engaged and informed about the progress and developments in this area, we encourage you to visit our website regularly and participate in upcoming public forums. Your active involvement is essential as we strive to enhance our environmental policies for the betterment of our community and future generations."
}
# Apply the function to each rows in the dataframe (each row represents a letter)
df['Result'] = df["Letter Text"].apply(lambda value: completion_iteration(value=value))
df

#output to csv
#previously I was exporting just the new column and joining in PBI, but it is struggling to parse and combine on larger runs so just re-outputting the whole csv and using that as the source
df.to_csv(output_path, index=False)
df

#ran 346min for 22k rows
#615 min for 32k rows

,ProjectId,Project Name,Comment Period Id,Comment Period Type,Comment Period Name,Comment Period Start Year,Author Name,Author ZipCode,Letter Id,Letter Sequence Number,Letter Text,Letter Submission Date,Form Set,Letter Type,Delivery Type,Result
3,1214,2016 Renewal of Existing Permits for Outfitter...,1476,Scoping,NaN,2016,AJ Myers,97754,1151680,1,Length of permit: I am in favor of permits be...,2016-02-25 22:43:32.0000000,NaN,Unique,CARA Web-portal,None
4,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,mark perkins,85048,1253492,1,"Hello, great to read about the 4 Forest Restor...",2016-07-23 00:11:36.0000000,NaN,Unique,CARA Web-portal,None
5,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,Stephen Clark,85381,1289130,2,"August 8, 2016\n\nAnnette Fredette, 4FRI Plann...",2016-08-08 07:00:00.0000000,NaN,Unique,CARA Web-portal,None
6,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,Fred Gaudet,85067,1295026,3,Also see attachment for pdf of letter\n\nAugus...,2016-08-09 19:59:00.0000000,NaN,Unique,CARA Web-portal,None
7,1356,4FRI Rim Country Project,1668,Scoping,NaN,2016,Nate Reisner,86001,1313126,4,"4 FRI Project Team: Annette Fredette, 4 FRI Pl...",2016-10-06 00:00:00.0000000,NaN,Unique,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366342,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,LeeAnn Bennett,66049-2029,1373906,2,"Forest Service,\n\nI have no objection to this...",2017-04-29 02:36:43.0000000,NaN,Unique,CARA Web-portal,None
366343,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,Scott Amos,NaN,1375429,3,It's not very often that I have something posi...,2017-04-28 11:00:00.0000000,NaN,Unique,Email,None
366345,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,Joel Fields,NaN,1375431,5,We have a summer cabin in Yellow Pine and have...,2017-05-01 11:00:00.0000000,NaN,Unique,Email,None
366346,1619,Yellow Pine Blowdown,2012,Scoping,NaN,2017,Johnna Sandow,NaN,1375432,6,I saw this has a June 2017 implementation date...,2017-04-28 11:00:00.0000000,NaN,Unique,Email,None


#Notes

Letter text referenced as:
df["Letter Text"]

Text response is referenced with: 
completion.choices[0].message.content

Takes about 500 tokens per row, priced at $.0015 for GPT-3.5 would be about $30 to run it on 2020-2021 letters (20k unique rows)
It took about 27 minutes to run on 1000 rows so would need to run over the weekend or on a VM 